# 25 prepare liana inputs

**Purpose:** Prepare cell groups and inputs for LIANA.

**Original analysis notebook:** `23_prepare_LIANA_CCC_inputs_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 42 — Prepare LIANA cell–cell communication inputs  
**Project:** GBM scRNA-seq, HVG8000, scVI, CopyKAT consensus  
**Goal:** prepare robust, thesis-ready inputs for LIANA/LIANA+ ligand–receptor analysis.

This notebook does **not** run LIANA yet. It prepares and audits:
- final cell-type labels for communication analysis,
- main vs exploratory CCC grouping,
- per-condition and per-sample cell-type counts,
- inclusion/exclusion decisions,
- lightweight metadata tables for the next LIANA notebook.

Why a preparation step?  
Cell–cell communication inference is sensitive to cell labels, rare groups, review/ambiguous populations, and sample imbalance. Here we define a conservative **main LIANA analysis** and a separate **exploratory review-inclusive analysis**.

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scanpy as sc

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURE_DIR = PROJECT_DIR / "figures" / "LIANA_HVG8000_r0_6_prepare"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = "LIANA_prepare_HVG8000_r0_6"

INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Required final malignant-state object not found: {INPUT_H5AD}. "
        "Run notebooks 17 and 20 first."
    )

MIN_TOTAL_CELLS_MAIN = 200
MIN_SAMPLES_MAIN = 3
MIN_CELLS_PER_CONDITION_FOR_CONDITION_ANALYSIS = 50

# To save disk space, this notebook does NOT write a duplicate h5ad by default.
# The next LIANA notebook can load the final h5ad and merge the exported metadata.
WRITE_LIANA_READY_H5AD = False

print("[INPUT_H5AD]", INPUT_H5AD)
print("[METADATA_DIR]", METADATA_DIR)

In [ ]:
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")
    return path

def choose_first_existing_column(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Could not find {label}. Tried: {candidates}. Available: {list(df.columns)}")
    return None

def clean_label(x):
    x = str(x)
    x = x.strip()
    x = x.replace("/", "_")
    x = x.replace("-", "_")
    x = x.replace(" ", "_")
    x = x.replace("__", "_")
    return x

def safe_mode(series):
    s = series.dropna().astype(str)
    if len(s) == 0:
        return np.nan
    return s.value_counts().index[0]

In [ ]:
print("[READ backed obs/var]")
adata_b = sc.read_h5ad(INPUT_H5AD, backed="r")
obs = adata_b.obs.copy()
var = adata_b.var.copy()
n_obs = adata_b.n_obs
n_vars = adata_b.n_vars

print(adata_b)
print("[obs columns]", len(obs.columns))
print(list(obs.columns)[:80])
print("[var columns]", list(var.columns)[:20])

In [ ]:
# Resolve key columns robustly.
cluster_col = choose_first_existing_column(
    obs,
    ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6", "leiden_scvi_0.6", "leiden", "cluster"],
    label="cluster column"
)

annotation_col = choose_first_existing_column(
    obs,
    ["final_annotation_refined", "annotation_marker_based_HVG8000_r0_6", "annotation_marker_based", "cell_type_marker_based", "annotation"],
    label="final annotation column"
)

compartment_col = choose_first_existing_column(
    obs,
    ["final_compartment_major", "final_malignancy_group_copykat_marker", "malignancy_group", "final_group"],
    label="final compartment column"
)

condition_col = choose_first_existing_column(
    obs,
    ["condition", "disease", "diagnosis"],
    label="condition column"
)

sample_col = choose_first_existing_column(
    obs,
    ["sample_id", "sample", "Sample"],
    label="sample column"
)

patient_col = choose_first_existing_column(
    obs,
    ["patient_id", "patient", "Patient"],
    required=False,
    label="patient column"
)

dataset_col = choose_first_existing_column(
    obs,
    ["dataset_id", "dataset", "core_dataset"],
    required=False,
    label="dataset column"
)

copykat_col = choose_first_existing_column(
    obs,
    ["copykat_pred_clean", "copykat_prediction", "copykat_call"],
    required=False,
    label="CopyKAT column"
)

malignant_state_col = choose_first_existing_column(
    obs,
    ["malignant_state", "malignant_state_final", "condition_malignant_state", "final_malignant_state"],
    required=False,
    label="malignant state column"
)

resolved_cols = pd.DataFrame([
    {"role": "cluster", "column": cluster_col},
    {"role": "annotation", "column": annotation_col},
    {"role": "compartment", "column": compartment_col},
    {"role": "condition", "column": condition_col},
    {"role": "sample", "column": sample_col},
    {"role": "patient", "column": patient_col},
    {"role": "dataset", "column": dataset_col},
    {"role": "copykat", "column": copykat_col},
    {"role": "malignant_state", "column": malignant_state_col},
])

write_tsv_replace(resolved_cols, METADATA_DIR / f"{OUT_PREFIX}_resolved_columns.tsv")
display(resolved_cols)

In [ ]:
# Standardize metadata values.
obs = obs.copy()
obs["_cell_id"] = obs.index.astype(str)
obs["_cluster"] = obs[cluster_col].astype(str)
obs["_annotation"] = obs[annotation_col].astype(str)
obs["_compartment"] = obs[compartment_col].astype(str)
obs["_condition"] = obs[condition_col].astype(str)
obs["_sample_id"] = obs[sample_col].astype(str)

if patient_col is not None:
    obs["_patient_id"] = obs[patient_col].astype(str)
else:
    obs["_patient_id"] = obs["_sample_id"]

if dataset_col is not None:
    obs["_dataset_id"] = obs[dataset_col].astype(str)
else:
    obs["_dataset_id"] = "unknown_dataset"

if copykat_col is not None:
    obs["_copykat"] = obs[copykat_col].astype(str)
else:
    obs["_copykat"] = "not_available"

if malignant_state_col is not None:
    obs["_malignant_state"] = obs[malignant_state_col].astype(str)
else:
    obs["_malignant_state"] = "not_available"

print(obs[["_cell_id", "_cluster", "_annotation", "_compartment", "_condition", "_sample_id", "_malignant_state"]].head())

In [ ]:
# Define LIANA cell-type labels.
# Main CCC analysis: malignant states + high-confidence non-malignant/diploid-leaning lineages.
# Review populations are excluded from main analysis, but retained in an exploratory label.

def assign_liana_label(row, include_review=False):
    cluster = str(row["_cluster"])
    ann = str(row["_annotation"])
    comp = str(row["_compartment"])
    state = str(row["_malignant_state"])
    ann_clean = clean_label(ann)

    comp_low = comp.lower()
    ann_low = ann.lower()
    state_low = state.lower()

    # Malignant/cycling states.
    if "cycling" in comp_low or "cycling" in state_low or cluster == "2":
        return "Malignant_cycling_candidate"

    if "malignant" in comp_low:
        if state not in ["not_available", "not_malignant_state", "nan", "None"]:
            s = clean_label(state)
            s = s.replace("condition_malignant_state_", "")
            s = s.replace("state_marker_", "")
            return f"Malignant_{s}"
        if cluster == "0":
            return "Malignant_OPC_NPC_like"
        if cluster == "4":
            return "Malignant_hypoxia_stress"
        if cluster == "7":
            return "Malignant_AC_reactive"
        if cluster == "11":
            return "Malignant_NPC_neuronal_like"
        return f"Malignant_{ann_clean}"

    # High-confidence non-malignant / diploid-leaning groups.
    if "non_malignant" in comp_low or "diploid" in comp_low:
        if "oligodendrocyte" in ann_low:
            return "Oligodendrocyte_myelinating"
        if "endothelial" in ann_low:
            return "Endothelial"
        if "mural" in ann_low or "pericyte" in ann_low or "fibroblast" in ann_low:
            return "Mural_Pericyte_Fibroblast"
        if "b_plasma" in ann_low or "plasma" in ann_low or ann_low.startswith("b_"):
            return "B_Plasma"
        if "mast" in ann_low:
            return "Mast_cell"
        return clean_label(ann)

    # Review/ambiguous groups.
    if include_review:
        # Keep review groups separated by cluster to avoid mixing unrelated signals.
        return f"Review_cluster_{cluster}_{ann_clean}"

    return np.nan

obs["liana_celltype_main"] = obs.apply(lambda r: assign_liana_label(r, include_review=False), axis=1)
obs["liana_celltype_exploratory"] = obs.apply(lambda r: assign_liana_label(r, include_review=True), axis=1)

print("[MAIN labels]")
display(obs["liana_celltype_main"].value_counts(dropna=False).reset_index().rename(columns={"index": "label", "liana_celltype_main": "n"}).head(50))

print("[EXPLORATORY labels]")
display(obs["liana_celltype_exploratory"].value_counts(dropna=False).reset_index().rename(columns={"index": "label", "liana_celltype_exploratory": "n"}).head(80))

In [ ]:
# Cell type counts.
def count_table(label_col):
    d = (
        obs.dropna(subset=[label_col])
        .groupby(label_col, observed=True)
        .size()
        .reset_index(name="n_cells")
        .rename(columns={label_col: "liana_celltype"})
        .sort_values("n_cells", ascending=False)
    )
    d["analysis_label"] = label_col
    return d

main_counts = count_table("liana_celltype_main")
exploratory_counts = count_table("liana_celltype_exploratory")
celltype_counts = pd.concat([main_counts, exploratory_counts], ignore_index=True)

write_tsv_replace(celltype_counts, METADATA_DIR / f"{OUT_PREFIX}_celltype_counts.tsv")
display(celltype_counts.head(80))

In [ ]:
# Sample-level and condition-level counts.
def grouped_counts(label_col):
    tmp = obs.dropna(subset=[label_col]).copy()
    sample_counts = (
        tmp.groupby(["_condition", "_dataset_id", "_patient_id", "_sample_id", label_col], observed=True)
        .size()
        .reset_index(name="n_cells")
        .rename(columns={
            "_condition": "condition",
            "_dataset_id": "dataset_id",
            "_patient_id": "patient_id",
            "_sample_id": "sample_id",
            label_col: "liana_celltype",
        })
    )

    condition_counts = (
        tmp.groupby(["_condition", label_col], observed=True)
        .size()
        .reset_index(name="n_cells")
        .rename(columns={"_condition": "condition", label_col: "liana_celltype"})
    )

    return sample_counts, condition_counts

main_sample_counts, main_condition_counts = grouped_counts("liana_celltype_main")
expl_sample_counts, expl_condition_counts = grouped_counts("liana_celltype_exploratory")

main_sample_counts["analysis_mode"] = "main"
main_condition_counts["analysis_mode"] = "main"
expl_sample_counts["analysis_mode"] = "exploratory"
expl_condition_counts["analysis_mode"] = "exploratory"

sample_counts = pd.concat([main_sample_counts, expl_sample_counts], ignore_index=True)
condition_counts = pd.concat([main_condition_counts, expl_condition_counts], ignore_index=True)

write_tsv_replace(sample_counts, METADATA_DIR / f"{OUT_PREFIX}_sample_celltype_counts.tsv")
write_tsv_replace(condition_counts, METADATA_DIR / f"{OUT_PREFIX}_condition_celltype_counts.tsv")

display(condition_counts.head(80))

In [ ]:
# Inclusion decisions for main LIANA analysis.
main_tmp = obs.dropna(subset=["liana_celltype_main"]).copy()

total_by_ct = main_tmp.groupby("liana_celltype_main", observed=True).size().rename("n_total_cells")
samples_by_ct = main_tmp.groupby("liana_celltype_main", observed=True)["_sample_id"].nunique().rename("n_samples_present")
conditions_by_ct = main_tmp.groupby("liana_celltype_main", observed=True)["_condition"].nunique().rename("n_conditions_present")

cond_pivot = (
    main_tmp.groupby(["liana_celltype_main", "_condition"], observed=True)
    .size()
    .unstack(fill_value=0)
)

decision = pd.concat([total_by_ct, samples_by_ct, conditions_by_ct], axis=1).reset_index().rename(columns={"liana_celltype_main": "liana_celltype"})
for cond in sorted(obs["_condition"].unique()):
    if cond in cond_pivot.columns:
        decision[f"n_cells_{cond}"] = decision["liana_celltype"].map(cond_pivot[cond]).fillna(0).astype(int)
    else:
        decision[f"n_cells_{cond}"] = 0

decision["include_main_global"] = (
    (decision["n_total_cells"] >= MIN_TOTAL_CELLS_MAIN) &
    (decision["n_samples_present"] >= MIN_SAMPLES_MAIN)
)

cond_cols = [c for c in decision.columns if c.startswith("n_cells_")]
decision["n_conditions_with_min_cells"] = (decision[cond_cols] >= MIN_CELLS_PER_CONDITION_FOR_CONDITION_ANALYSIS).sum(axis=1)
decision["include_condition_specific"] = decision["n_conditions_with_min_cells"] >= 2

decision["note"] = np.where(
    decision["include_main_global"],
    "included in main LIANA global analysis",
    "excluded from main LIANA global analysis due to low cells or low sample support"
)

decision = decision.sort_values(["include_main_global", "n_total_cells"], ascending=[False, False])
write_tsv_replace(decision, METADATA_DIR / f"{OUT_PREFIX}_celltype_inclusion_decisions.tsv")
display(decision)

In [ ]:
# Export per-cell metadata for next LIANA notebook.
cols_to_export = [
    "_cell_id",
    "_condition",
    "_dataset_id",
    "_patient_id",
    "_sample_id",
    "_cluster",
    "_annotation",
    "_compartment",
    "_malignant_state",
    "_copykat",
    "liana_celltype_main",
    "liana_celltype_exploratory",
]

cell_meta = obs[cols_to_export].copy()
cell_meta = cell_meta.rename(columns={
    "_cell_id": "cell_id",
    "_condition": "condition",
    "_dataset_id": "dataset_id",
    "_patient_id": "patient_id",
    "_sample_id": "sample_id",
    "_cluster": "cluster",
    "_annotation": "final_annotation_refined",
    "_compartment": "final_compartment_major",
    "_malignant_state": "malignant_state",
    "_copykat": "copykat_pred_clean",
})

included_main_types = set(decision.loc[decision["include_main_global"], "liana_celltype"])
cell_meta["liana_include_main_global"] = cell_meta["liana_celltype_main"].isin(included_main_types)
cell_meta["liana_include_exploratory"] = cell_meta["liana_celltype_exploratory"].notna()

write_tsv_replace(cell_meta, METADATA_DIR / f"{OUT_PREFIX}_cell_metadata.tsv")

main_cells = cell_meta[cell_meta["liana_include_main_global"]].copy()
expl_cells = cell_meta[cell_meta["liana_include_exploratory"]].copy()

write_tsv_replace(main_cells, METADATA_DIR / f"{OUT_PREFIX}_main_cells.tsv")
write_tsv_replace(expl_cells, METADATA_DIR / f"{OUT_PREFIX}_exploratory_cells.tsv")

print("[main cells]", main_cells.shape)
print("[exploratory cells]", expl_cells.shape)
display(main_cells.head())

In [ ]:
# Readiness and manifest.
readiness = pd.DataFrame([
    {"item": "input_h5ad", "value": str(INPUT_H5AD), "status": "found"},
    {"item": "n_cells", "value": int(n_obs), "status": "ok"},
    {"item": "n_genes", "value": int(n_vars), "status": "ok"},
    {"item": "cluster_col", "value": cluster_col, "status": "ok"},
    {"item": "annotation_col", "value": annotation_col, "status": "ok"},
    {"item": "compartment_col", "value": compartment_col, "status": "ok"},
    {"item": "condition_col", "value": condition_col, "status": "ok"},
    {"item": "sample_col", "value": sample_col, "status": "ok"},
    {"item": "patient_col", "value": patient_col if patient_col else "sample_id used as patient_id", "status": "ok"},
    {"item": "dataset_col", "value": dataset_col if dataset_col else "unknown_dataset", "status": "ok"},
    {"item": "copykat_col", "value": copykat_col if copykat_col else "not_available", "status": "ok"},
    {"item": "malignant_state_col", "value": malignant_state_col if malignant_state_col else "not_available", "status": "ok"},
    {"item": "main_liana_celltypes", "value": int(main_counts.shape[0]), "status": "ok"},
    {"item": "main_cells_included_global", "value": int(main_cells.shape[0]), "status": "ok"},
    {"item": "exploratory_cells", "value": int(expl_cells.shape[0]), "status": "ok"},
    {"item": "write_duplicate_liana_ready_h5ad", "value": bool(WRITE_LIANA_READY_H5AD), "status": "configured"},
    {"item": "created_at", "value": datetime.now().isoformat(timespec="seconds"), "status": "ok"},
])

readiness_path = METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv"
write_tsv_replace(readiness, readiness_path)
display(readiness)

manifest_records = []
for p in [
    METADATA_DIR / f"{OUT_PREFIX}_resolved_columns.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_sample_celltype_counts.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_condition_celltype_counts.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_inclusion_decisions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_cell_metadata.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_main_cells.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_exploratory_cells.tsv",
    readiness_path,
]:
    manifest_records.append({
        "path": str(p),
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / 1024 / 1024, 3) if p.exists() else np.nan,
    })

manifest = pd.DataFrame(manifest_records)
manifest_path = METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv"
write_tsv_replace(manifest, manifest_path)
display(manifest)

## What to send back after running this notebook

Please send these files:

```text
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_prepare_HVG8000_r0_6_readiness.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_prepare_HVG8000_r0_6_resolved_columns.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_prepare_HVG8000_r0_6_celltype_counts.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_prepare_HVG8000_r0_6_sample_celltype_counts.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_prepare_HVG8000_r0_6_condition_celltype_counts.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_prepare_HVG8000_r0_6_celltype_inclusion_decisions.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_prepare_HVG8000_r0_6_output_manifest.tsv
```

Do not send the full h5ad.